In [1]:


import sys, os

# # silence output temporarily 
# sys.stdout, sys.stderr = os.devnull, os.devnull

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.spatial.distance import cdist
from sklearn.preprocessing import OneHotEncoder
import matplotlib.colors as mcolors
import matplotlib.image as mpimg

import scipy 
import figure_utils as utils
# 

from sklearn import manifold
from mcbo.utils.experiment_utils import get_task_from_id

from ssp_bayes_opt import agents

import pygraphviz as pgv
import networkx as nx
import grakel
from grakel.kernels import WeisfeilerLehman
import nengo

import pickle
import json
import grakel
# unsilence output
# sys.stdout, sys.stderr = sys.__stdout__, sys.__stderr__

%matplotlib inline

np.random.seed(0)

In [2]:


def generate_smooth_trajectory(num_points=100, dim=2, domain_bounds=np.array([[-1, 1], [-1, 1]]), step_size=0.2):
    num_dense_points = num_points * 10  # Generate a dense random walk
    
    # Start at a random point within the domain
    start_point = np.array([np.random.uniform(low, high) for low, high in domain_bounds])
    traj = [start_point]
    
    for _ in range(num_dense_points - 1):
        step = np.random.uniform(-step_size, step_size, size=dim)  # Random step
        next_point = traj[-1] + step
        
        # Keep trajectory within bounds
        next_point = np.clip(next_point, domain_bounds[:, 0], domain_bounds[:, 1])
        traj.append(next_point)
    
    traj = np.array(traj)
    
    # Down-sample to num_points
    indices = np.linspace(0, num_dense_points - 1, num_points, dtype=int)
    traj_downsampled = traj[indices]
    
    # Apply Gaussian smoothing
    traj_smoothed = scipy.ndimage.gaussian_filter1d(traj_downsampled, sigma=2, axis=0)
    
    return traj_smoothed

# Generate reference trajectory
num_points = 10
dim = 2
traj_reference = generate_smooth_trajectory(num_points, dim)
# traj_reference[:,0] -= np.mean(traj_reference[:,0])
# traj_reference[:,1] -= np.mean(traj_reference[:,1])
shift_fun = lambda x, axis, new_min, new_max: (new_max - new_min)*(x - np.min(x,axis=axis,keepdims=True))/(np.max(x,axis=axis,keepdims=True) - np.min(x,axis=axis,keepdims=True))  + new_min

# def shift_fun(x, axis, new_min=-0.8, new_max=0.8):
#     sample_shape = np.sum(x,axis=axis,keepdims=True).shape
#     new_min = np.maximum(-0.99, np.random.normal(new_min, 0.5, size=sample_shape))
#     new_max = np.minimum(0.99, np.random.normal(new_max, 0.5, size=sample_shape))
#     res = (new_max - new_min)*(x - np.min(x,axis=axis,keepdims=True))
#     res = res/(np.max(x,axis=axis,keepdims=True) - np.min(x,axis=axis,keepdims=True))  + new_min
#     return res

def shift_fun(x, axis, new_max=0.8):
    x = x - np.expand_dims(x.take(0, axis=axis), axis=axis)
    # np.put(x, -1, np.max(x,axis=axis,keepdims=True)
    x = new_max*x/np.max(np.sqrt(np.sum(x**2,axis=-1,keepdims=True)),axis=axis,keepdims=True)
    return x

# traj_reference = shift_fun(traj_reference,0, -0.9,0.9) 

# Generate 100 random trajectories
num_trajectories = 1000
trajectories = np.array([generate_smooth_trajectory(num_points, dim) for _ in range(num_trajectories)])
# trajectories = shift_fun(trajectories, 1, -0.9,0.9)
trajectories = shift_fun(trajectories, 1, 0.9)
traj_reference = shift_fun(traj_reference, 0, 0.9)

trajectories = np.vstack([trajectories, traj_reference[None,:,:]])

plt.figure()
plt.plot(traj_reference[:,0], traj_reference[:,1], '.-', marker='*')
for i in range(1,5):
    plt.plot(trajectories[i,:,0], trajectories[i,:,1], '.-')


In [7]:
## FROM https://github.com/spiros/discrete_frechet/blob/master/frechetdist.py
def _c(ca, i, j, p, q):

    if ca[i, j] > -1:
        return ca[i, j]
    elif i == 0 and j == 0:
        ca[i, j] = np.linalg.norm(p[i]-q[j])
    elif i > 0 and j == 0:
        ca[i, j] = max(_c(ca, i-1, 0, p, q), np.linalg.norm(p[i]-q[j]))
    elif i == 0 and j > 0:
        ca[i, j] = max(_c(ca, 0, j-1, p, q), np.linalg.norm(p[i]-q[j]))
    elif i > 0 and j > 0:
        ca[i, j] = max(
            min(
                _c(ca, i-1, j, p, q),
                _c(ca, i-1, j-1, p, q),
                _c(ca, i, j-1, p, q)
            ),
            np.linalg.norm(p[i]-q[j])
            )
    else:
        ca[i, j] = float('inf')

    return ca[i, j]


def frdist(p, q):
    """
    Computes the discrete Fréchet distance between
    two curves. The Fréchet distance between two curves in a
    metric space is a measure of the similarity between the curves.
    The discrete Fréchet distance may be used for approximately computing
    the Fréchet distance between two arbitrary curves,
    as an alternative to using the exact Fréchet distance between a polygonal
    approximation of the curves or an approximation of this value.

    This is a Python 3.* implementation of the algorithm produced
    in Eiter, T. and Mannila, H., 1994. Computing discrete Fréchet distance.
    Tech. Report CD-TR 94/64, Information Systems Department, Technical
    University of Vienna.
    http://www.kr.tuwien.ac.at/staff/eiter/et-archive/cdtr9464.pdf

    Function dF(P, Q): real;
        input: polygonal curves P = (u1, . . . , up) and Q = (v1, . . . , vq).
        return: δdF (P, Q)
        ca : array [1..p, 1..q] of real;
        function c(i, j): real;
            begin
                if ca(i, j) > −1 then return ca(i, j)
                elsif i = 1 and j = 1 then ca(i, j) := d(u1, v1)
                elsif i > 1 and j = 1 then ca(i, j) := max{ c(i − 1, 1), d(ui, v1) }
                elsif i = 1 and j > 1 then ca(i, j) := max{ c(1, j − 1), d(u1, vj) }
                elsif i > 1 and j > 1 then ca(i, j) :=
                max{ min(c(i − 1, j), c(i − 1, j − 1), c(i, j − 1)), d(ui, vj ) }
                else ca(i, j) = ∞
                return ca(i, j);
            end; /* function c */

        begin
            for i = 1 to p do for j = 1 to q do ca(i, j) := −1.0;
            return c(p, q);
        end.

    Parameters
    ----------
    P : Input curve - two dimensional array of points
    Q : Input curve - two dimensional array of points

    Returns
    -------
    dist: float64
        The discrete Fréchet distance between curves `P` and `Q`.

    Examples
    --------
    >>> from frechetdist import frdist
    >>> P=[[1,1], [2,1], [2,2]]
    >>> Q=[[2,2], [0,1], [2,4]]
    >>> frdist(P,Q)
    >>> 2.0
    >>> P=[[1,1], [2,1], [2,2]]
    >>> Q=[[1,1], [2,1], [2,2]]
    >>> frdist(P,Q)
    >>> 0
    """
    p = np.array(p, np.float64)
    q = np.array(q, np.float64)

    len_p = len(p)
    len_q = len(q)

    if len_p == 0 or len_q == 0:
        raise ValueError('Input curves are empty.')
        
    ca = (np.ones((len_p, len_q), dtype=np.float64) * -1)

    dist = _c(ca, len_p-1, len_q-1, p, q)
    return dist

# #"a measure of similarity between curves that takes into account the location and ordering of the points along the curves


# Compute distance measure (mean absolute distance between waypoints)
# distances = [np.mean(np.linalg.norm(traj_reference - traj, axis=1)) for traj in trajectories]
#distances = [frdist(traj_reference , traj) for traj in trajectories]


ssp_agt = agents.SSPTrajectoryAgent(trajectories[:10,:,:].reshape(10,-1),
                                    np.zeros((10,1)),
                                    x_dim=2,
                                    traj_len=num_points, optim_ls=False,
                                   length_scale=0.25, ssp_dim = 500, 
                                   decoder_method='from-set') 
all_tssps = ssp_agt.encode(trajectories.reshape(num_trajectories+1,-1))


In [4]:
def hexagonal_packing(R, hex_radius):
    """
    Generates the (x, y) locations of the centers of hexagons in a hexagonal packing inside a circle of radius R.
    :param R: Radius of the circular boundary
    :param hex_radius: Radius of each hexagon
    :return: List of (x, y) coordinates
    """
    coords = []
    dx = 3/2 * hex_radius  # Horizontal distance between centers of adjacent hexagons
    dy = np.sqrt(3) * hex_radius  # Vertical distance between centers of adjacent rows
    
    # Estimate number of hexagon rows required
    max_rows = int(R / dy) + 1  # Extra to ensure full coverage
    
    for row in range(-max_rows, max_rows + 1):
        y = row * dy
        row_offset = (row % 2) * (dx / 2)  # Offset alternate rows
        max_cols = int((R - abs(y)) / dx) + 1  # Limit hexagons in the row based on y-position
        
        for col in range(-max_cols, max_cols + 1):
            x = col * dx + row_offset
            
            # Check if inside the circle
            if x**2 + y**2 <= R**2:
                coords.append((x, y))
    
    return coords

# Example usage
R = 10  # Circle radius
hex_radius = 1  # Hexagon radius
hex_centers = hexagonal_packing(R, hex_radius)

# Plot the results
# x_vals, y_vals = zip(*hex_centers)
# plt.figure(figsize=(8, 8))
# plt.scatter(x_vals, y_vals, s=100, color='blue', label='Hex Centers')

# # Draw the circle boundary
# circle = plt.Circle((0, 0), R, color='r', fill=False, linestyle='--')
# plt.gca().add_patch(circle)
# plt.xlim(-R-1, R+1)
# plt.ylim(-R-1, R+1)
# plt.gca().set_aspect('equal')
# plt.legend()
# plt.title('Hexagonal Packing in a Circle')
# plt.show()


In [5]:
cmap_colors = [(1,1,1), (0,0,0), (1,1,1)]
cmap_positions = [0.0,0.5,1.0]
gray_cmap = mcolors.LinearSegmentedColormap.from_list("cyclic_gray", list(zip(cmap_positions,cmap_colors)))
norm_map = plt.Normalize(- np.pi, np.pi)

hex_pack_centers = np.array(hexagonal_packing(ssp_agt.ssp_dim*0.2, 10))

def plot_ssp_phasors(ax,ssp_space,ssp,radius=1,scale=1, headwidth=2, width=0.02):
    d=len(ssp)
    
    circle_locations = hex_pack_centers
    idxs = np.arange(0, np.minimum(d//2,circle_locations.shape[0]), 1)
    print(circle_locations.shape)
    print(d//2)
    #nengo.dists.ScatteredHypersphere(surface=False).sample(d, 2)[::2,:]
    #ssp_space.get_sample_points(d//2,method='sobol')
    # phase_matrix[1:(d//2)+1,:]
    circle_locations = circle_locations[idxs]
    circle_locations = (circle_locations - np.min(circle_locations))/(np.max(circle_locations) - np.min(circle_locations))
    circle_locations = 15*(2*circle_locations - 1) 

    ssp = ssp.flatten()/np.linalg.norm(ssp)
    complex_numbers = np.fft.fft(ssp[idxs])[1:(d//2)+1]
    # print(complex_numbers.shape)
    complex_numbers = complex_numbers/np.sqrt(complex_numbers.real ** 2 + complex_numbers.imag ** 2)
    # print(complex_numbers)
    angles = np.angle(complex_numbers)
    for i, (c_num, angle) in enumerate(zip(complex_numbers, angles)):
        loc = circle_locations[i]  # Position for this subplot
        
        color = gray_cmap(norm_map(angle))
        inv_color = np.array(color)
        inv_color[:-1] = 0 if inv_color[0] > 0.5 else 1
    
        quiver=ax.quiver(loc[0], loc[1], c_num.real, c_num.imag,  angles='xy', 
                              scale_units='xy', color=inv_color, zorder=2,scale=scale, headwidth=headwidth, width=width,alpha=0.7,clip_on=False)
        circle = plt.Circle(loc, radius, color=color, fill=True, zorder=1, linewidth=0,alpha=0.7,clip_on=False)
        ax.add_patch(circle)


# fig,ax = plt.subplots(1,1,figsize=(2,2))
# plot_ssp_phasors(ax, ssp_agt.ssp_x_space, all_tssps[0,:],scale=1,radius=1,headwidth=1,width=0.01)

In [92]:

# Compute kernel values using an RBF kernel
def rbf_kernel(traj1, traj2, gamma=1.0):
    squared_dist = np.sum((traj1 - traj2) ** 2)
    return np.exp(-gamma * squared_dist)

def ssp_kernel(traj1, traj2, ssp_agt):
    ssp1 = ssp_agt.encode(traj1.reshape(1,-1))
    ssp2 = ssp_agt.encode(traj2.reshape(1,-1))
    sim = np.sum(ssp1 * ssp2)/(np.linalg.norm(ssp1)*np.linalg.norm(ssp2))
    return sim

# distances = [frdist(traj_reference , traj) for traj in trajectories] 

distances = [np.mean(np.linalg.norm(traj_reference - traj, axis=1)) for traj in trajectories]
kernel_values = np.array([ssp_kernel(traj_reference, traj, ssp_agt) for traj in trajectories])
kernel_values = kernel_values/np.max(kernel_values)

rbf_kernel_values = np.array([rbf_kernel(traj_reference, traj, gamma=0.5) for traj in trajectories])

# rbf_kernel_values = rbf_kernel_values/np.max(rbf_kernel_values)

plt.figure()
plt.scatter(distances,kernel_values,5)

In [61]:
def complex_to_torus(z1_real, z1_imag, z2_real, z2_imag, R=2, r=1):
    theta_1 = np.arctan2(z1_imag, z1_real)  # Angle of first complex number
    theta_2 = np.arctan2(z2_imag, z2_real)  # Angle of second complex number

    # Compute torus coordinates
    r1=np.sqrt(z1_imag**2 + z1_real**2)
    r2=np.sqrt(z2_imag**2 + z2_real**2)
    # print(r1,r2)
    x = (R + r * np.cos(theta_2)) * np.cos(theta_1)
    y = (R + r * np.cos(theta_2)) * np.sin(theta_1)
    z = r * np.sin(theta_2)

    return x, y, z


all_tssps2 = np.fft.fft(all_tssps,axis=-1)#ssp_agt.ssp_x_space.make_unitary(all_tssps)
p_x,p_y,p_z = complex_to_torus(all_tssps2[:,1].real,all_tssps2[:,1].imag,all_tssps2[:,2].real,all_tssps2[:,2].imag, R=1.5)
fig=plt.figure(figsize=(3,3))
# ax = fig.add_subplot()
ax = fig.add_subplot(projection='3d')
ax.scatter(p_x,p_y,p_z,s=2)

fig=plt.figure(figsize=(3,3))
plt.imshow(all_tssps[0,:400].reshape(20,20))

In [93]:
n_neighbors=10
n_components = 3

params = {
    "n_neighbors": n_neighbors,
    "n_components": n_components,
    "eigen_solver": "auto",
    "random_state": 0,
}

# t_sne = manifold.TSNE(
#     n_components=n_components,
#     perplexity=10,
#     init="random",
#     random_state=0,
# )
# low_dim_embed = t_sne.fit_transform(all_tssps)

lle_standard = manifold.LocallyLinearEmbedding(method="standard", **params)
low_dim_embed = lle_standard.fit_transform(all_tssps)

# isomap = manifold.Isomap(n_neighbors=n_neighbors, n_components=n_components, p=1)
# low_dim_embed = isomap.fit_transform(all_tssps)

# md_scaling = manifold.MDS(
#     n_components=n_components,
#     max_iter=50,
#     n_init=4,
#     random_state=0,
#     normalized_stress=False,
# )
# low_dim_embed = md_scaling.fit_transform(all_tssps)

# spectral = manifold.SpectralEmbedding(
#     n_components=n_components, n_neighbors=n_neighbors, random_state=42
# )
# low_dim_embed = spectral.fit_transform(all_tssps)

fig= plt.figure()
if n_components==2:
    plt.scatter(low_dim_embed[:,0], low_dim_embed[:,1], 
            c=kernel_values,
            s=50, alpha=0.8)
elif n_components==3:
    ax = fig.add_subplot(projection='3d')
    ax.scatter(low_dim_embed[:,0], low_dim_embed[:,1], low_dim_embed[:,2],
             c=kernel_values,
             s=10, alpha=0.8, linewidth=0.)
    # ax.set_axis_off()
    for old_lim, set_fun in zip([ ax.get_xlim(), ax.get_ylim(), ax.get_zlim()],
                                [ ax.set_xlim, ax.set_ylim, ax.set_zlim]):
        set_fun(old_lim[0]*0.6, old_lim[1]*0.6)
    

plt.show()

In [155]:
#### MCBO combin vars


def overlap_kernel(x1, x2, diag=False, lengthscale=1.0,ard_num_dims=None):
    """Implementation of the categorical overlap kernel 
    This is the most basic form of the categorical kernel that essentially invokes a Kronecker delta function
    between any two elements.
    """
    # First, convert one-hot to ordinal representation
    diff = x1[:, None] - x2[None, :]
    
    # nonzero location = different category
    diff[np.abs(diff) > 1e-5] = 1
    
    # invert to count same categories
    diff1 = np.logical_not(diff).astype(float)
    
    if ard_num_dims is not None and ard_num_dims > 1:
        k_cat = np.sum(lengthscale * diff1, axis=-1) / np.sum(lengthscale)
    else:
        # dividing by number of categorical variables to keep this term in range [0,1]
        k_cat = np.sum(diff1, axis=-1) / x1.shape[1]
    
    if diag:
        return np.diag(k_cat)
    
    return k_cat

def get_samples(num_points,search_space,):
    df_samples = search_space.sample(num_points)
    for p in search_space.params:
        df_samples[p] = search_space.params[p].transform(df_samples[p]).cpu().numpy()
    return np.array(df_samples).astype(float)


mcbo_task = get_task_from_id("rna_inverse_fold")
search_space = mcbo_task.get_search_space()
n_params = len(search_space.params)
reference_sample = get_samples(1, search_space)
cat_im_size = int(np.sqrt(len(reference_sample.flatten())))
reference_sample = reference_sample[:,:cat_im_size**2]

samples = get_samples(1000, search_space)
samples = samples[:,:cat_im_size**2]
samples = np.vstack([reference_sample,samples])
overlap_kernel_values = overlap_kernel(reference_sample,samples)

search_space.nominal_names = search_space.nominal_names[:cat_im_size**2]

mcbo_agt = agents.SSPMCBOAgent(samples[:5,:], np.ones((5,1)),
                               search_space, ssp_dim=501)
all_combin_ssps = mcbo_agt.encode(samples)


mcbo_kernel_values = np.array([ssp_kernel(reference_sample, s, mcbo_agt) for s in samples])
# mcbo_kernel_values = mcbo_kernel_values/np.max(mcbo_kernel_values)

fig=plt.figure(figsize=(2,2))
plt.scatter(1-overlap_kernel_values,mcbo_kernel_values)


# one_hot_ref = np.zeros((n_params,np.max([search_space.params[p].ub+1 for p in search_space.params])))
# one_hot_ref[np.arange(n_params),reference_sample.flatten().astype(int)] = 1.0

# plt.figure(figsize=(2,2))
# plt.imshow(one_hot_ref)

plt.figure()
plt.imshow(reference_sample[:,:cat_im_size**2].reshape(cat_im_size,cat_im_size), cmap='Set2')

n_neighbors=2
n_components = 3

params = {
    "n_neighbors": n_neighbors,
    "n_components": n_components,
    "eigen_solver": "auto",
    "random_state": 0,
}

# t_sne = manifold.TSNE(
#     n_components=n_components,
#     perplexity=1,
#     init="random",
#     random_state=0,
# )
# low_dim_embed_mcbo = t_sne.fit_transform(all_combin_ssps)




lle_standard = manifold.LocallyLinearEmbedding(method="standard", **params)
low_dim_embed_mcbo = lle_standard.fit_transform(all_combin_ssps)

# isomap = manifold.Isomap(n_neighbors=n_neighbors, n_components=n_components, p=1)
# low_dim_embed = isomap.fit_transform(all_combin_ssps)

# md_scaling = manifold.MDS(
#     n_components=n_components,
#     max_iter=50,
#     n_init=4,
#     random_state=0,
# )
# low_dim_embed_mcbo = md_scaling.fit_transform(all_combin_ssps)

# spectral = manifold.SpectralEmbedding(
#     n_components=n_components, n_neighbors=n_neighbors, random_state=42
# )
# low_dim_embed_mcbo = spectral.fit_transform(all_combin_ssps)

fig= plt.figure()
if n_components==2:
    plt.scatter(low_dim_embed_mcbo[:,0], low_dim_embed_mcbo[:,1], 
            c=overlap_kernel_values.flatten(),
            s=50, alpha=0.8)
elif n_components==3:
    ax = fig.add_subplot(projection='3d')
    ax.scatter(low_dim_embed_mcbo[:,0], low_dim_embed_mcbo[:,1], low_dim_embed_mcbo[:,2],
             c=overlap_kernel_values.flatten(),
             s=10, alpha=0.8, linewidth=0.)
    # ax.set_axis_off()
    for old_lim, set_fun in zip([ ax.get_xlim(), ax.get_ylim(), ax.get_zlim()],
                                [ ax.set_xlim, ax.set_ylim, ax.set_zlim]):
        set_fun(old_lim[0]*0.6, old_lim[1]*0.6)
    

plt.show()

In [110]:
from grakel.kernels import WeisfeilerLehman, VertexHistogram
from grakel.kernels import RandomWalkLabeled

def draw_network_NAS(g, path, backbone=False,dpi=300):
    graph = pgv.AGraph(directed=True, strict=True, fontname='cmunbmr.ttf', arrowtype='open',
                      fontpath='/Users/nicoledumont/Library/Fonts', dpi=dpi)
    # graph = pgv.AGraph(directed=True, strict=True, fontname='SourceCodePro', arrowtype='open')

    
    adjlist=[list(g.predecessors(i)) for i in range(len(g))]+ [list(g.successors(len(g)-2))]
    if g is None:
        add_node_NAS(graph, 0, 0)
        graph.layout(prog='dot')
        graph.draw(path)
        return
    for idx in range(len(g)):
        add_node_NAS(graph, idx, nx.get_node_attributes(g, 'Label')[idx])
    for idx in range(len(g)):
        for node in adjlist[idx]:
            if node == idx-1 and backbone:
                graph.add_edge(node, idx, weight=1)
            else:
                graph.add_edge(node, idx, weight=0)
    graph.layout(prog='dot')
    graph.draw(path)

def add_node_NAS(graph, node_id, label, shape='box', style='filled'):
    if label == 0:  
        label = 'output'
        color=utils.purples[2]#'#bd93f9'
    elif label == 1:
        label = 'input'
        color = utils.blues[2]#'#ff79c6'    
    elif label == 2:
        label = 'conv1'
        color = utils.yellows[2]#'#66d9ef'
    elif label == 3:
        label = 'conv3'
        color = utils.oranges[2]#'#ffb86c'
    elif label == 4:
        label = 'max3'
        color = utils.greens[2]# '#6272a4'
    else:
        label = ''
        color = utils.blues[2]#'aliceblue'
    label = f"{label}"
    graph.add_node(
            node_id, label=label, color='black', fillcolor=color,
            shape=shape, style=style, fontsize=9,
            **{'width':'0', 'height':'0'})


def parse_graph_to_nx(e, label, flat=False):
    try:
        G=nx.DiGraph()
        for i in range(e.shape[1]):
            G.add_edge(e[0][i].item(),e[1][i].item())
        for i in range(len(G)):
            G.nodes[i]['Label']= label[i].item()
    except: 
        G=nx.empty_graph()
    if flat==True:
        adj_flatten=np.array([int(s) for s in e.split(' ')])
        nodes=([int(s) for s in label.split(' ')])
        n=len(nodes)
        matrix=adj_flatten.reshape(n,n)
        G=nx.DiGraph()
        edges=np.transpose(np.transpose(np.nonzero(matrix)))        
        for i in range(edges.shape[1]):
            G.add_edge(edges[0][i].item(),edges[1][i].item())
        for i in range(len(G)):
            G.nodes[i]['Label']= nodes[i]
    return G

with open("nas_full_plot_data.pkl", 'rb') as input_file:
   loaded_graph_data = pickle.load(input_file)

with open("nas_graph_plot_data.json") as input_file:
   string_graph_data = np.array(json.load(input_file))

ref_graph = grakel.Graph(initialization_object=loaded_graph_data[0][0],
             node_labels={i: label for i, label in enumerate(loaded_graph_data[0][1])},
                         graph_format='adjacency')
ref_graph_ssp = loaded_graph_data[0][2]


all_graphs = [
    grakel.Graph(initialization_object=data[0], 
                 node_labels={i: label for i, label in enumerate(data[1])},
                 graph_format='adjacency')
    for data in loaded_graph_data  
]

all_graph_ssps = np.array([data[2].flatten() for data in loaded_graph_data])

graph_ssp_sims =  (all_graph_ssps @ ref_graph_ssp.T)/(np.linalg.norm(ref_graph_ssp)*np.linalg.norm(all_graph_ssps,axis=1,keepdims=True))
#np.array([ssp_kernel(ref_graph_ssp, s, mcbo_agt) for s in all_graph_ssps])
# all_graph_ssps @ ref_graph_ssp.T
# graph_ssp_sims = graph_ssp_sims/np.max(graph_ssp_sims)


# kernel = WeisfeilerLehman(normalize=True, base_graph_kernel=VertexHistogram)
# graph_similarities = kernel.fit_transform(all_graphs)


kernel = RandomWalkLabeled(normalize=True)  
kernel.fit([ref_graph])
graph_similarities = kernel.transform(all_graphs)


plt.figure(figsize=(2,1))
plt.scatter(graph_similarities[1:],graph_ssp_sims.flatten()[1:],s=1)


n_neighbors=5
n_components = 3

params = {
    "n_neighbors": n_neighbors,
    "n_components": n_components,
    "eigen_solver": "auto",
    "random_state": 0,
}


lle_standard = manifold.LocallyLinearEmbedding(method="standard", **params)
graph_dim_embed = lle_standard.fit_transform(all_graph_ssps)

fig= plt.figure()
if n_components==2:
    plt.scatter(graph_dim_embed[:,0], graph_dim_embed[:,1], 
            c=graph_ssp_sims.flatten(),
            s=50, alpha=0.8)
elif n_components==3:
    ax = fig.add_subplot(projection='3d')
    ax.scatter(graph_dim_embed[:,0], graph_dim_embed[:,1], graph_dim_embed[:,2],
             c=graph_ssp_sims.flatten(),
             s=10, alpha=0.8, linewidth=0.)
    # ax.set_axis_off()
    for old_lim, set_fun in zip([ ax.get_xlim(), ax.get_ylim(), ax.get_zlim()],
                                [ ax.set_xlim, ax.set_ylim, ax.set_zlim]):
        set_fun(old_lim[0]*0.6, old_lim[1]*0.6)
    

plt.show()

In [207]:
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1.inset_locator import mark_inset, zoomed_inset_axes
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.patches import ConnectionPatch, Rectangle
from matplotlib import patches

cols = [utils.blues[0], utils.reds[0], utils.oranges[0], utils.greens[0], utils.purples[0], utils.yellows[0]]


def plot_traj(ax, x, scale_units='xy', angles='xy', scale=1, **kwargs):
    xs = x[::2,0]
    ys = x[::2,1]
    dxs = xs[1:]-xs[:-1]
    dys = ys[1:]-ys[:-1]
    ax.quiver(xs[:-1], ys[:-1], dxs, dys, 
              scale_units=scale_units, angles=angles, scale=scale, **kwargs)

def plot_cats(ax,x, color='Set2',fontsize=7, show_dif=False, ref=None, **kwargs):
    x = np.atleast_2d(x)
    grid_values = x[:, :cat_im_size**2].reshape(cat_im_size, cat_im_size)
    
    ax.pcolormesh(np.arange(cat_im_size),np.arange(cat_im_size),
       grid_values, cmap=color, **kwargs)

    if show_dif:
        grid_ref = ref[0,:cat_im_size**2].reshape(cat_im_size, cat_im_size)
        cells_to_outline = np.where(grid_ref==grid_values)
        for i in range(len(cells_to_outline[0])):
            x0, x1 = cells_to_outline[1][i]-0.5, cells_to_outline[1][i] +0.5
            y0, y1 = cells_to_outline[0][i]-0.5, cells_to_outline[0][i] + 0.5
            ax.plot([x0, x1, x1, x0, x0], [y0, y0, y1, y1, y0], color='k', linewidth=0.5)

    
    for i in range(cat_im_size):
        for j in range(cat_im_size):
            if show_dif:
                if grid_ref[i,j]==grid_values[i,j]:
                    color='k'
                else: 
                    color='grey'
            else:
                color='k'
            ax.text(j , i , str(int(grid_values[i, j])), 
                    ha="center", va="center", color=color, fontsize=fontsize)#, fontweight="bold")


def plot_graph(ax,graph_str,pad=0,**kwargs):
    g=parse_graph_to_nx(graph_str[0], graph_str[1], flat=True)
    draw_network_NAS(g, './_temp.png',dpi=600)
    img = mpimg.imread("_temp.png")
    img = np.pad(img, ((pad, pad), (pad, pad), (0,0)), constant_values=1.)

    ax.imshow(img, interpolation="none", resample=False)
    # ax.set_axis_off()
    

def make_row(fig,G, plot_sample_fun, 
             reference,samples,
             low_dim_embed,
             distances, kernel_values,
             bounds = [-1,1],
             annotate='', annotate_xy=(0,0), 
             annotate_k='',kernel_x = '',annotate_phi='',
             poss=[[0.5, 0.9, 0.5,0.5], [1.2, 0.5, 0.5,0.5]],
         inset_box_w = [0.025,0.05],
             main_sample_col=None, other_sample_cols=None,
             plot_ontop=True,
             plot_phasors=False,ssp_space=None,ref_ssp=None,
            plot_sample_args={}, plot_subsample_args={}, plot_phasors_args={},
            vis_zoom_lines=[[0,1],[0,1]],
            inset_bounds=None,
             annotate_phi_xytext=(20, -20),
            inds=[10,-200]):

    if inset_bounds is None:
        inset_bounds=bounds
    
    plot_subsample_args = plot_sample_args | plot_subsample_args
    gs0 = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=G, 
                                           width_ratios=[2,1 if plot_phasors else 2,6], 
                                           wspace=0.4 if plot_phasors else 0.5)
    
    ax0 = fig.add_subplot(gs0[0])
    if plot_phasors:
        ax1 = fig.add_subplot(gs0[1])
    else:
        ax1 = fig.add_subplot(gs0[1], projection='3d')
    ax2 = fig.add_subplot(gs0[2])
    axes = [ax0,ax1,ax2]

    # for i in range(len(axes) - 1):
    #     # Get positions in figure coordinates
    #     bbox1 = axes[i].get_position()
    #     bbox2 = axes[i+1].get_position()
    
    #     # Coordinates for the arrow (outside the axes)
    #     if i==0:
    #         dx1 = 0.01
    #         dx2 = 0.01
    #     else:
    #         dx1 = 0.0
    #         dx2 = 0.05
    #     x_start = bbox1.x1 + dx1  # Right edge of the current subplot
    #     x_end = bbox2.x0 - dx2    # Left edge of the next subplot
    #     y_mid = (bbox1.y0 + bbox1.y1) / 2  # Vertical center
    
    #     # Create arrow
    #     arrow = patches.FancyArrowPatch(
    #         (x_start, y_mid), (x_end, y_mid),
    #         transform=fig.transFigure,  # Use figure coordinates
    #         connectionstyle="arc3,rad=0",  # Straight arrow
    #         arrowstyle='->',
    #         mutation_scale=20,  # Arrow size
    #         lw=2,
    #         color='black'
    #     )
    
    #     # Add arrow to figure
    #     fig.add_artist(arrow)
    
    plot_sample_fun(ax0, reference, color=main_sample_col,
             **plot_sample_args)
    ax0.annotate(annotate,xy=annotate_xy,
                    xytext=(2, -20),  
                    textcoords='offset points',
                    ha='center', va='bottom', clip_on=False)
    
    ax0.set_axis_off()
    ax0.set_xlim(bounds[0])
    ax0.set_ylim(bounds[1])
    ax0.set_aspect('equal','box')

    if plot_phasors:
        plot_ssp_phasors(ax1, ssp_space, ref_ssp, **plot_phasors_args)
        for old_lim, set_fun in zip([ ax1.get_xlim(), ax1.get_ylim()],
                                    [ ax1.set_xlim, ax1.set_ylim]):
            set_fun(old_lim[0]*0.8, old_lim[1]*0.8)
        ax1.annotate(annotate_phi,xy=(8,-8),
                    xytext=(2, -20),  
                    textcoords='offset points',
                    ha='center', va='bottom',clip_on=False)
    else:
        ax1.scatter(low_dim_embed[:,0], low_dim_embed[:,1], low_dim_embed[:,2],
                 c=kernel_values,zorder=1,
                 s=5, alpha=0.3, linewidth=0., clip_on=False)
        ax1.plot(low_dim_embed[0,0], low_dim_embed[0,1], low_dim_embed[0,2],
                    color='k', markersize=6,marker='x',markeredgewidth=1.5,
                 zorder=3,alpha=1.0, clip_on=False) 
        for old_lim, set_fun in zip([ ax1.get_xlim(), ax1.get_ylim(), ax1.get_zlim()],
                                    [ ax1.set_xlim, ax1.set_ylim, ax1.set_zlim]):
            set_fun(old_lim[0]*0.4, old_lim[1]*0.4)

        ax1.annotate(annotate_phi,xy=(low_dim_embed[0,0], low_dim_embed[0,1]),
                    xytext=annotate_phi_xytext,  
                    textcoords='offset points',
                    ha='center', va='bottom', clip_on=False)
    ax1.set_axis_off()
    ax1.set_aspect('equal', 'box')
    
    ax2.scatter(distances[1:], kernel_values[1:], s=2, marker='.', color='k',zorder=1,alpha=0.8) 
    ax2.set_xlabel(kernel_x) #
    ax2.set_ylabel(annotate_k)
    
    sort_idx = np.argsort(distances)
    idx1 = sort_idx[inds[0]]
    idx2 = sort_idx[inds[1]]
    for i, (idx, pos) in enumerate(zip([idx1, idx2], 
                                       poss)):
        axin = ax2.inset_axes(pos, transform=ax2.transData)
        axin.set_aspect('equal', 'box')
        _,indlines = ax2.indicate_inset([distances[idx]-inset_box_w[0]/2, kernel_values[idx]-inset_box_w[1]/2,
                            inset_box_w[0],inset_box_w[1]],
                           axin,zorder=5,alpha=1)
        for j in range(4):
            if j in vis_zoom_lines[i]:
                indlines[j].set_visible(True)
            else:
                indlines[j].set_visible(False)
        axin.get_xaxis().set_ticks([])
        axin.get_yaxis().set_ticks([])
        axin.spines[['right', 'top']].set_visible(True)
        
        if plot_ontop:
            plot_sample_fun(axin, reference, color=main_sample_col,
                     alpha=0.2, **plot_sample_args)
        plot_sample_fun(axin, samples[idx], color=other_sample_cols(i),
                 **plot_subsample_args)

        axin.set_xlim(inset_bounds[0])
        axin.set_ylim(inset_bounds[1])


fig = plt.figure(figsize=(7.5,5),dpi=1000)

Gs = gridspec.GridSpec(3,1, figure=fig, hspace=1) # will add more rows later


make_row(fig,Gs[0],plot_traj,
         traj_reference,trajectories,
         low_dim_embed,
         distances, kernel_values,
         bounds = [[-0.5,1],[-1,0.5]],
         inset_bounds=[[-0.5,1],[-1,1]],
         annotate='$\\tau=\{(t_i, \\mathbf{x}(t_i))\}_i$',
         annotate_xy = (traj_reference[-1,0]+0.5, traj_reference[-1,1]),
         annotate_k = 'Kernel $k(\\tau, \\cdot)$',
         kernel_x='L$_2$ Distance',
         poss=[[0.3, 0.8, 0.5,0.5],
               [0.8, 0.6, 0.5,0.5]],
         inset_box_w = [0.025,0.05],
         main_sample_col='k', plot_ontop=True,
         other_sample_cols = lambda i:utils.greens[0] if i==0 else utils.reds[0], 
         plot_sample_args={'width':0.02,'headwidth':5,'scale':1},
        plot_phasors=False, ssp_space=ssp_agt.ssp_x_space,ref_ssp=all_tssps[0],
         plot_phasors_args={'scale':1,'radius':1,'headwidth':1,'width':0.01},annotate_phi='$\\phi(\\tau)$',
        vis_zoom_lines=[[0,1],[0,2]], annotate_phi_xytext=(10,-20))


# make_row(fig,Gs[1],plot_cats,
#          reference_sample,samples,
#          low_dim_embed_mcbo,
#          1-overlap_kernel_values.flatten(), mcbo_kernel_values,
#          bounds = [[-0.5,cat_im_size-0.5],[-0.5,cat_im_size-0.5]],
#          annotate='$C$',
#          annotate_xy = (4.5,0.1),
#          annotate_k = 'Kernel $k(C, \\cdot)$',
#          kernel_x='Overlap Similarity',
#          poss=[[0.5, 0.4, 0.3,0.3], 
#                [0.8, 0.2, 0.3,0.3]],
#          inset_box_w = [0.01,0.05],
#          main_sample_col='Set2', 
#          plot_ontop=False,
#          other_sample_cols = lambda i: 'Set2', 
#          plot_sample_args={},plot_subsample_args={'fontsize':3,
#                                                   'show_dif': True, 'ref': reference_sample}, 
#          plot_phasors=False, ssp_space=None,ref_ssp=all_combin_ssps[0],
#          plot_phasors_args={'scale':1,'radius':1,'headwidth':1,'width':0.01},annotate_phi='$\\phi(C)$',
#         vis_zoom_lines=[[0,1],[1,2]])


make_row(fig,Gs[1],plot_cats,
         reference_sample,samples,
         low_dim_embed_mcbo,
         overlap_kernel_values.flatten(), mcbo_kernel_values,
         bounds = [[-0.5,cat_im_size-0.5],[-0.5,cat_im_size-0.5]],
         annotate='$C$',
         annotate_xy = (4.5,0.1),
         annotate_k = 'Kernel $k(C, \\cdot)$',
         kernel_x='Overlap Similarity',
         poss=[[-0.1, 0.27, 0.4,0.4], 
               [0.22, 0.53, 0.4,0.4]],
         inset_box_w = [0.01,0.05],
         main_sample_col='Set2', 
         plot_ontop=False,
         other_sample_cols = lambda i: 'Set2', 
         plot_sample_args={},plot_subsample_args={'fontsize':3,
                                                  'show_dif': True, 'ref': reference_sample}, 
         plot_phasors=False, ssp_space=None,ref_ssp=all_combin_ssps[0],
         plot_phasors_args={'scale':1,'radius':1,'headwidth':1,'width':0.01},annotate_phi='$\\phi(C)$',
        vis_zoom_lines=[[0,2],[0,2]],
        inds=[30, -20], annotate_phi_xytext=(10,-30))

make_row(fig,Gs[2],plot_graph,
         string_graph_data[0],string_graph_data,
         graph_dim_embed,
         graph_similarities.flatten(), graph_ssp_sims.flatten(),
         bounds = [None,None],#,[[0,2000],[0,2000]],
         inset_bounds = [None,None],#[[-300,1600],[-100,2200]],
         annotate='$G$',
         annotate_xy = (900,900),
         annotate_k = 'Kernel $k(G, \\cdot)$',
         kernel_x='Random Walk Similarity',
         poss=[[0.91, 0.45, 0.07,0.7], 
               [0.95, 0.75, 0.07,0.7]],
         inset_box_w = [0.001,0.05],
         main_sample_col=None, 
         plot_ontop=False,
         other_sample_cols = lambda i: None, 
         plot_sample_args={},plot_subsample_args={'pad':100}, 
         plot_phasors=False, 
         vis_zoom_lines=[[0,3],[0,3]],
         annotate_phi='$\\phi(G)$',
        inds=[20, -6], annotate_phi_xytext=(10,-30))
#plot_phasors=True, ssp_space=mcbo_agt.ssp_x_space,

fig.text(0.11,0.95,"\\textbf{a} $\quad$ Actions", fontsize=9)
fig.text(0.31,0.95,"\\textbf{b} $\quad$ VSA Representations", fontsize=9)
fig.text(0.62,0.95,"\\textbf{c} $\quad$ VSA Kernel Functions", fontsize=9)


fig.text(0.11,0.9,"\\textbf{i} $\quad$ Trajectories", fontsize=9)
fig.text(0.11,0.63,"\\textbf{ii} $\quad$ Multivariate Categories", fontsize=9)
fig.text(0.11,0.3,"\\textbf{iii} $\quad$ Graphs", fontsize=9)

utils.save(fig, "embed_kernel_compare2.pdf")
# fig

In [189]:
utils.save(fig, "embed_kernel_compare2.png")
